In [19]:
import pandas as pd
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV
import joblib


In [20]:
#Cargamos los datos
X_train = pd.read_csv("../data/2. processed/X_train.csv")
X_test = pd.read_csv("../data/2. processed/X_test.csv")
y_train = pd.read_csv("../data/2. processed/y_train.csv")
y_test = pd.read_csv("../data/2. processed/y_test.csv")

In [21]:
xgb_base= XGBClassifier( n_estimators=100, random_state=42)

In [22]:
xgb_base.fit(X_train,y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=100,
              n_jobs=None, num_parallel_tree=None, ...)

In [23]:
train_pred = xgb_base.predict(X_train)
test_pred = xgb_base.predict(X_test)

In [24]:
print("Train")
print(classification_report(y_train, train_pred))
print("--------------------------------------------------")
print("Test")
print(classification_report(y_test, test_pred))

Train
              precision    recall  f1-score   support

           0       0.90      0.88      0.89     25337
           1       0.80      0.82      0.81     14663

    accuracy                           0.86     40000
   macro avg       0.85      0.85      0.85     40000
weighted avg       0.86      0.86      0.86     40000

--------------------------------------------------
Test
              precision    recall  f1-score   support

           0       0.82      0.81      0.81      6334
           1       0.68      0.70      0.69      3666

    accuracy                           0.77     10000
   macro avg       0.75      0.75      0.75     10000
weighted avg       0.77      0.77      0.77     10000



In [25]:
joblib.dump(xgb_base, "../models/smoking_model_xgb.pkl")


['../models/smoking_model_xgb.pkl']

Optimización XGBoost

In [26]:
param_grid_xgb = {
    "n_estimators": [100, 200],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.05, 0.1],
    "subsample": [0.8, 1.0]
}

In [27]:
grid_xgb = GridSearchCV(
    estimator=XGBClassifier( n_estimators=100, random_state=42),
    param_grid=param_grid_xgb,
    scoring="f1",
    cv=5,
    n_jobs=4,
    verbose=2
)

In [28]:
grid_xgb.fit(X_train, y_train)


Fitting 5 folds for each of 24 candidates, totalling 120 fits


GridSearchCV(cv=5,
             estimator=XGBClassifier(base_score=None, booster=None,
                                     callbacks=None, colsample_bylevel=None,
                                     colsample_bynode=None,
                                     colsample_bytree=None, device=None,
                                     early_stopping_rounds=None,
                                     enable_categorical=False, eval_metric=None,
                                     feature_types=None, feature_weights=None,
                                     gamma=None, grow_policy=None,
                                     importance_type=None,
                                     interaction_constraints=Non...
                                     max_cat_to_onehot=None,
                                     max_delta_step=None, max_depth=None,
                                     max_leaves=None, min_child_weight=None,
                                     missing=nan, monotone_constraints=None,
                                     multi_strategy=None, n_estimators=100,
                                     n_jobs=None, num_parallel_tree=None, ...),
             n_jobs=4,
             param_grid={'learning_rate': [0.05, 0.1], 'max_depth': [3, 5, 7],
                         'n_estimators': [100, 200], 'subsample': [0.8, 1.0]},
             scoring='f1', verbose=2)

In [29]:
best_xgb = grid_xgb.best_estimator_# Con esto extraemos los "mejores" hiperparámetros encontrados por GridSearchCV

In [30]:
train_pred_xgb = best_xgb.predict(X_train) 
test_pred_xgb = best_xgb.predict(X_test)

In [31]:
print("Train - best")
print(classification_report(y_train, train_pred_xgb))
print("--------------------------------------------------")
print("Test -best")
print(classification_report(y_test, test_pred_xgb))

Train - best
              precision    recall  f1-score   support

           0       0.89      0.86      0.87     25337
           1       0.77      0.81      0.79     14663

    accuracy                           0.84     40000
   macro avg       0.83      0.84      0.83     40000
weighted avg       0.84      0.84      0.84     40000

--------------------------------------------------
Test -best
              precision    recall  f1-score   support

           0       0.83      0.80      0.82      6334
           1       0.68      0.73      0.70      3666

    accuracy                           0.77     10000
   macro avg       0.76      0.76      0.76     10000
weighted avg       0.78      0.77      0.78     10000



In [32]:
#Guardamos el mejor modelo entrenado.
joblib.dump(best_xgb, "../models/smoking_model_xgb_best.pkl")


['../models/smoking_model_xgb_best.pkl']